# 02 — RFM Hesaplama ve Skorlama
Bu notebook `data_clean.csv` dosyasını okur, RFM metriklerini hesaplar ve müşterileri segmentlere ayırır.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data_clean.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Referans tarihi: veri setinin son gününden 1 gün sonrası
REFERENCE_DATE = pd.Timestamp('2011-12-10')

# Her satırın toplam tutarı
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print(f'Temiz veri: {len(df):,} satır, {df["CustomerID"].nunique():,} müşteri')
df.head()

Temiz veri: 387,841 satır, 4,338 müşteri


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [2]:
# ── RFM METRİKLERİNİ HESAPLA ─────────────────────────────
# Recency  : Referans tarihinden son alışverişe kadar geçen gün sayısı
# Frequency: Benzersiz fatura (sipariş) sayısı
# Monetary : Toplam harcama tutarı (£)

rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (REFERENCE_DATE - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

rfm['Monetary'] = rfm['Monetary'].round(2)

print('RFM Tablosu:')
print(rfm.describe().round(2))
rfm.head(10)

RFM Tablosu:
       CustomerID  Recency  Frequency   Monetary
count     4338.00  4338.00    4338.00    4338.00
mean     15300.41    92.06       4.27    1753.95
std       1721.81   100.01       7.70    6315.14
min      12346.00     0.00       1.00       3.75
25%      13813.25    17.00       1.00     292.63
50%      15299.50    50.00       2.00     640.80
75%      16778.75   141.75       5.00    1567.46
max      18287.00   373.00     209.00  225198.80


,CustomerID,Recency,Frequency,Monetary
0,12346,325,1,124.80
1,12347,2,7,4185.20
2,12348,75,4,1328.88
3,12349,18,1,1443.80
4,12350,310,1,309.40
5,12352,36,8,1505.74
6,12353,204,1,89.00
7,12354,232,1,1073.55
8,12355,214,1,459.40
9,12356,22,3,2757.43


In [3]:
# ── RFM SKORLAMA (1-5 Quantile) ──────────────────────────
# Her metrik 5 eşit dilime ayrılır ve 1-5 arası skor alır
# Recency: düşük gün sayısı daha iyi → ters sıralama (5=en iyi)
# Frequency ve Monetary: yüksek değer daha iyi → normal sıralama

rfm['R_Score'] = pd.qcut(rfm['Recency'],
                          q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# rank(method='first') → eşit değerlerde sıralama hatası önlenir
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'),
                          q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['M_Score'] = pd.qcut(rfm['Monetary'],
                          q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Toplam skor (3-15 arası)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# 3 basamaklı segment kodu (örn. 555 = şampiyon, 111 = kayıp)
rfm['RFM_Segment'] = (rfm['R_Score'].astype(str)
                    + rfm['F_Score'].astype(str)
                    + rfm['M_Score'].astype(str))

print('RFM Skorları:')
rfm[['CustomerID','Recency','Frequency','Monetary',
     'R_Score','F_Score','M_Score','RFM_Score','RFM_Segment']].head(10)

RFM Skorları:


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Segment
0,12346,325,1,124.80,1,1,1,3,111
1,12347,2,7,4185.20,5,5,5,15,555
2,12348,75,4,1328.88,2,4,4,10,244
3,12349,18,1,1443.80,4,1,4,9,414
4,12350,310,1,309.40,1,1,2,4,112
5,12352,36,8,1505.74,3,5,4,12,354
6,12353,204,1,89.00,1,1,1,3,111
7,12354,232,1,1073.55,1,1,4,6,114
8,12355,214,1,459.40,1,1,2,4,112
9,12356,22,3,2757.43,4,3,5,12,435


In [4]:
# ── MÜŞTERİ SEGMENTİ ETİKETLEME ─────────────────────────
# RFM skorlarına göre her müşteriye iş anlamı taşıyan bir etiket atanır

def rfm_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    score   = row['RFM_Score']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Şampiyonlar'        # sık alır, çok harcar, yakın zamanda
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Sadık Müşteriler'   # tutarlı alışveriş yapıyor
    elif r >= 4 and f <= 2:
        return 'Yeni Müşteriler'    # yakın zamanda geldi ama henüz az alışveriş
    elif r >= 3 and f >= 2 and m >= 2:
        return 'Potansiyel Sadıklar' # sadık olmaya aday
    elif r <= 2 and f >= 3 and m >= 3:
        return 'Risk Altındakiler'  # eskiden iyiydi, uzaklaşıyor
    elif r <= 2 and f >= 4 and m >= 4:
        return 'Kaybedilmek Üzere' # değerliydi ama çok uzaklaştı
    elif r <= 1 and f <= 2 and m <= 2:
        return 'Kayıp Müşteriler'   # çok eskide kaldı, az harcadı
    elif score >= 9:
        return 'Umut Vaad Edenler'
    else:
        return 'Pasif Müşteriler'

rfm['Segment'] = rfm.apply(rfm_segment, axis=1)

# Segment özet tablosu
seg_summary = rfm.groupby('Segment').agg(
    Müşteri_Sayısı = ('CustomerID', 'count'),
    Ort_Recency    = ('Recency',    'mean'),
    Ort_Frequency  = ('Frequency',  'mean'),
    Ort_Monetary   = ('Monetary',   'mean'),
).round(1).sort_values('Müşteri_Sayısı', ascending=False)

print('Segment Dağılımı:')
print(seg_summary)

print('\nToplam gelirin segmentlere dağılımı:')
total = rfm['Monetary'].sum()
for seg_name, grp in rfm.groupby('Segment'):
    pct = grp['Monetary'].sum() / total * 100
    print(f'  {seg_name:<22}: %{pct:.1f}')

Segment Dağılımı:
                     Müşteri_Sayısı  Ort_Recency  Ort_Frequency  Ort_Monetary
Segment                                                                      
Pasif Müşteriler                987        126.9            1.4         421.1
Şampiyonlar                     952         12.2           11.1        5279.9
Sadık Müşteriler                769         35.0            4.1        1475.8
Kayıp Müşteriler                554        279.3            1.0         213.2
Risk Altındakiler               457        140.6            3.8        1369.8
Yeni Müşteriler                 312         17.8            1.2         401.1
Potansiyel Sadıklar             276         37.3            2.0         477.6
Umut Vaad Edenler                31         24.1            2.3         985.7

Toplam gelirin segmentlere dağılımı:
  Kayıp Müşteriler      : %1.6
  Pasif Müşteriler      : %5.5
  Potansiyel Sadıklar   : %1.7
  Risk Altındakiler     : %8.2
  Sadık Müşteriler      : %14.9
  Umut V

In [5]:
# Kaydet
rfm.to_csv('rfm_segments.csv', index=False)
print('✅ rfm_segments.csv kaydedildi')
rfm.head()

✅ rfm_segments.csv kaydedildi


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Segment,Segment
0,12346,325,1,124.80,1,1,1,3,111,Kayıp Müşteriler
1,12347,2,7,4185.20,5,5,5,15,555,Şampiyonlar
2,12348,75,4,1328.88,2,4,4,10,244,Risk Altındakiler
3,12349,18,1,1443.80,4,1,4,9,414,Yeni Müşteriler
4,12350,310,1,309.40,1,1,2,4,112,Kayıp Müşteriler
